# Tutorial: from oscillation probabilities to numbers of events

In this notebook we use a **simple model of the number of $\nu_e$ appearance events** at T2K, Hyper-K, NOvA and DUNE
(see `docs/number_of_events_summary.pdf`):

$$
N = P(\nu_\mu\to\nu_e;\,E_0, L)\;\sigma_{CC}(E_0)\;\Phi\;\mathrm{POT}\;N_T
  \;=\; \underbrace{N_{P=1}}_{\text{fixed for each experiment}}\;\times\;P(E_0) .
$$

This is the number of $\nu_e$ ($\bar\nu_e$) **charged-current interactions** in the detector, under three simple
assumptions: the beam is a pure $\nu_\mu$ ($\bar\nu_\mu$) beam, all neutrinos have the same energy $E_0$, and the detector
sees every interaction (efficiency $= 100\,\%$). In Section 4 we compare it with the numbers predicted by the
collaborations, which include the real detector and beam.

Everything except $P$ is fixed by the experiment (beam, detector, exposure). So **the number of events changes only
through the oscillation probability**: to see how $N$ depends on $\delta_{CP}$, on the mass ordering or on $\theta_{23}$,
we only need to recompute $P$ at the beam energy $E_0$.

Plan:
1. The inputs and the function `n_events`
2. Examples
3. Exercises: numbers of events, bi-probability and bi-event plots, sensitivity to CP violation
4. How realistic is the simple model? Comparison with the experiments' predictions

Solutions: `tutorial_event_counting_solutions.ipynb`. Try first!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nuosc import NeutrinoOscillator
from nuosc.plotting import set_style

SET_STYLE = True     # True: nuosc style (latex_style -> SciencePlots -> default); False: keep your own matplotlib settings
c = set_style(SET_STYLE)

## 1. The inputs and the function

For each experiment and beam mode (`"nu"`: neutrino beam, `"nubar"`: antineutrino beam) we need only two numbers,
taken from the table of `docs/number_of_events_summary.pdf`:

* `E0`: the beam energy [GeV] at which $P$ is evaluated;
* `N1`: $N_{P=1} = \sigma_{CC}(E_0)\,\Phi\,\mathrm{POT}\,N_T$, the number of CC interactions if every $\nu_\mu$ became a $\nu_e$.

The baseline $L$ and the matter density are those of the experiment configuration in `nuosc`
(`NeutrinoOscillator.from_experiment`).

In [ ]:
INPUTS = {
    #          beam mode: E0 [GeV], N_{P=1}
    "T2K":    {"nu": dict(E0=0.57, N1=994),   "nubar": dict(E0=0.55, N1=251)},
    "HyperK": {"nu": dict(E0=0.57, N1=37329), "nubar": dict(E0=0.55, N1=25809)},
    "NOvA":   {"nu": dict(E0=1.80, N1=5792),  "nubar": dict(E0=1.79, N1=975)},
    "DUNE":   {"nu": dict(E0=2.31, N1=47441), "nubar": dict(E0=2.27, N1=17384)},
}
EXPERIMENTS = list(INPUTS)


def probability(exp, beam, ordering="NO", **params):
    # P(numu -> nue) (beam = "nu") or P(numubar -> nuebar) (beam = "nubar") at the beam energy E0
    osc = NeutrinoOscillator.from_experiment(exp, ordering=ordering, antineutrino=(beam == "nubar"), **params)
    return float(osc.matter("mu", "e", E=INPUTS[exp][beam]["E0"]))


def n_events(exp, beam, ordering="NO", **params):
    # expected number of nue (nuebar) CC interactions = N_{P=1} * P(E0)
    return INPUTS[exp][beam]["N1"] * probability(exp, beam, ordering, **params)

`**params` are any oscillation parameters accepted by `NeutrinoOscillator` (`delta_cp` in degrees, `s2_23`, `dm2_31`, ...);
those not given are at the NuFIT 6.0 best fit of the chosen ordering.

## 2. Examples

In [ ]:
# number of events at the NuFIT 6.0 best fit (NO: delta_CP = 212 deg)
print(f"T2K, nu beam:    N = {n_events('T2K', 'nu'):.1f}")
print(f"T2K, nubar beam: N = {n_events('T2K', 'nubar'):.1f}")

# the same with another delta_CP: only P changes
for d in (0, 90, 180, 270):
    print(f"delta_CP = {d:3d} deg: P = {probability('T2K', 'nu', delta_cp=d):.4f},  N = {n_events('T2K', 'nu', delta_cp=d):.1f}")

# inverted ordering (its own NuFIT best fit) and another theta_23
print(f"T2K, nu beam, IO:            N = {n_events('T2K', 'nu', ordering='IO'):.1f}")
print(f"T2K, nu beam, sin2_23 = 0.6: N = {n_events('T2K', 'nu', s2_23=0.6):.1f}")

The statistical uncertainty on a number of events is $\sqrt N$:

In [ ]:
N = n_events("NOvA", "nu")
print(f"NOvA nu beam: N = {N:.1f} +- {np.sqrt(N):.1f}  (relative error {1 / np.sqrt(N):.0%})")

## 3. Exercises

**Exercise 1: number of events.** For the four experiments, compute the number of events in the $\nu$ and $\bar\nu$ beams
at the NuFIT 6.0 best fit, for NO and IO. Which experiment has the most events? In which beam?

**Exercise 2: bi-probability plot.** Fix all oscillation parameters to the NuFIT 6.0 normal-ordering best fit
(`NUFIT_NO` in `nuosc.parameters`) and compare two hypotheses that differ **only in the sign of $\Delta m^2_{31}$**:
NO ($\Delta m^2_{31}>0$) and IO ($\Delta m^2_{31}\to-\Delta m^2_{31}$). For each experiment and hypothesis, scan
$\delta_{CP}$ from $0$ to $360°$ and plot $P(\bar\nu_\mu\to\bar\nu_e)$ (y) against $P(\nu_\mu\to\nu_e)$ (x) (one panel
per experiment, one colour per hypothesis). Mark $\delta_{CP}=0$ and $90°$ on each ellipse. Then mark, with a single
black point, the "true" universe: NO with $\delta_{CP}=212°$ (the NuFIT best fit).
*Hint:* `probability(exp, "nu", "IO", **dict(NUFIT_NO, dm2_31=-NUFIT_NO["dm2_31"], delta_cp=d))`.

**Exercise 3: bi-event plot.** The same with the numbers of events $N_{\bar\nu}$ against $N_\nu$. The black point is now
a pseudo-measurement (Asimov data set): the expected numbers of events in the true universe, with error bars
$\pm\sqrt N$. Why do the bi-probability and bi-event plots have the same shape? Which points of the two ellipses are
compatible with the measurement? Which experiments can tell NO from IO, if the truth is NO with $\delta_{CP}=212°$?

**Exercise 4: sensitivity to CP violation.** For a true value of $\delta_{CP}$, the $\chi^2$ to reject CP conservation is
$$
\Delta\chi^2(\delta_{CP}) = \min_{\delta'\in\{0,\,180°\}} \sum_{\nu,\bar\nu}
\frac{\left[N(\delta_{CP}) - N(\delta')\right]^2}{N(\delta_{CP})},
$$
and the significance is $\sqrt{\Delta\chi^2}$ (in units of $\sigma$). Plot $\sqrt{\Delta\chi^2}$ against the true $\delta_{CP}$
for the four experiments, NO and IO. For which values of $\delta_{CP}$ is the sensitivity largest, and why is it zero at
$0$ and $180°$? Which experiments reach $3\sigma$ and $5\sigma$?

**Exercise 5: systematics.** Add a normalization uncertainty of 5 % on each sample, i.e. replace the denominator with
$N(\delta_{CP}) + (0.05\,N(\delta_{CP}))^2$, and compare with Exercise 4. Which experiments are affected, and why?

### Exercise 1

In [ ]:
# your code here

### Exercise 2

In [ ]:
# your code here

### Exercise 3

In [ ]:
# your code here

### Exercise 4

In [ ]:
# your code here

### Exercise 5

In [ ]:
# your code here

## 4. How realistic is the simple model? Comparison with the experiments' predictions

The collaborations publish the number of $\nu_e$ ($\bar\nu_e$) appearance events expected in their analyses, computed
with full simulations of the beam, of the cross sections and of the detector. Each prediction is given for a specific set
of oscillation parameters, so we evaluate our formula at the same parameters and take the ratio
$$
r = \frac{N_\text{published}}{N_{P=1}\,P(E_0)} .
$$
If our model were exact, $r$ would be 1.

In [ ]:
def s2(sin2_2theta):
    # sin^2(theta) from sin^2(2 theta), first octant
    return (1 - np.sqrt(1 - sin2_2theta)) / 2

REFERENCE = {
    # T2K, arXiv:2101.03779: Table XI total single-ring e-like events at delta_CP = -90 deg (74.4, 17.1) minus the
    # delta_CP-independent background (19.6, 12.3) from a fit of the four delta_CP values of the table; Table III parameters
    "T2K": dict(N=(54.8, 4.8), params=dict(s2_12=0.304, s2_13=0.0212, s2_23=0.528, delta_cp=270,
                                           dm2_21=7.53e-5, dm2_31=2.509e-3 + 7.53e-5)),
    # Hyper-K Design Report, arXiv:1805.04163, Tables XXXVI-XXXVII (10 years, 1:3 nu:nubar)
    "HyperK": dict(N=(1643, 1183), params=dict(s2_12=s2(0.8704), s2_13=s2(0.10), s2_23=0.5, delta_cp=0,
                                               dm2_21=7.6e-5, dm2_31=2.4e-3 + 7.6e-5)),
    # NOvA, arXiv:2509.04361, Table 2, at the NOvA best fit
    "NOvA": dict(N=(125.3, 18.9), params=dict(s2_12=s2(0.851), s2_13=s2(0.0851), s2_23=0.547, delta_cp=0.87 * 180,
                                              dm2_21=7.53e-5, dm2_31=2.441e-3 + 7.53e-5)),
    # DUNE, official GLoBES configuration (arXiv:2103.04797), signal at NuFIT 4.0 NO, delta_CP = 0
    "DUNE": dict(N=(2180, 438), params=dict(s2_12=np.sin(0.5903) ** 2, s2_13=np.sin(0.150) ** 2,
                                            s2_23=np.sin(0.866) ** 2, delta_cp=0, dm2_21=7.39e-5,
                                            dm2_31=2.451e-3 + 7.39e-5)),
}

RATIO = {}
print(f"{'':7s} {'beam':6s} {'published':>10s} {'N1 x P':>8s} {'ratio r':>8s}")
for exp, ref in REFERENCE.items():
    for k, beam in enumerate(("nu", "nubar")):
        N_simple = n_events(exp, beam, "NO", **ref["params"])
        RATIO[exp, beam] = ref["N"][k] / N_simple
        print(f"{exp:7s} {beam:6s} {ref['N'][k]:10.1f} {N_simple:8.1f} {RATIO[exp, beam]:8.2f}")

**Questions.** Why is $r<1$ for the $\nu$ beams? Why can it be close to or larger than 1 for some $\bar\nu$ beams?
What does this mean for the numbers of events and the significances of Exercises 1, 4 and 5?

### Section 4: your answers

In [ ]:
# your code here (optional)